# 01 · Limpeza e qualidade dos dados

**Responsável:** Francisco Figueiredo  ·  **Frente de trabalho:** Preparação dos dados

---

### Objetivo:
Produzir o **dataset limpo que todas os outros vão usar**

### O que este notebook entrega ao resto do grupo:
O dataset limpo.


## 0 · Setup

Importar as bibliotecas.

In [3]:
import pandas as pd
import numpy as np

## 1 · Primeira leitura

Olhar para a tabela antes de lhe tocar: dimensões, nomes e tipos das colunas, primeiras linhas,
estatísticas descritivas das numéricas e das categóricas.


In [12]:
raw_df = pd.read_csv('../data/raw/artist_streams.csv', index_col = 0)
raw_df

raw_df.info()                    # tipos de cada coluna e quantos não-nulos
raw_df.describe()                # min, max, média — apanha anos absurdos
raw_df.isna().sum()              # nulos por coluna
raw_df.duplicated().sum()        # linhas totalmente repetidas
raw_df['Country of Origin'].value_counts(dropna=False)   # categorias escritas de formas diferentes
raw_df.head()

<class 'pandas.DataFrame'>
Index: 511 entries, 0 to 500
Data columns (total 14 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Artist Name            510 non-null    str    
 1   Sex                    486 non-null    str    
 2   Country of Origin      510 non-null    str    
 3   Primary Language       510 non-null    str    
 4   Primary Genre          510 non-null    str    
 5   Artist Type            510 non-null    str    
 6   Debut Year             510 non-null    float64
 7   Total Streams          505 non-null    float64
 8   Lead Streams           504 non-null    float64
 9   Feature Streams        499 non-null    float64
 10  Solo Streams           506 non-null    float64
 11  Collaborative Streams  509 non-null    float64
 12  Update Date            510 non-null    str    
 13  Year                   510 non-null    float64
dtypes: float64(7), str(7)
memory usage: 91.0 KB


,Artist Name,Sex,Country of Origin,Primary Language,Primary Genre,Artist Type,Debut Year,Total Streams,Lead Streams,Feature Streams,Solo Streams,Collaborative Streams,Update Date,Year
Artist ID,,,,,,,,,,,,,,
0,Drake,Male,Canada,English,Hip-Hop,Solo,2006.0,137492.1,94851.1,42640.9,53323.9,84168.2,2026-07-01 00:00:00,2026.0
1,Taylor Swift,Female,United States,English,Pop,Solo,2006.0,127861.0,126177.3,1683.8,113421.5,14439.5,2026-07-01 00:00:00,2026.0
2,Bad Bunny,Male,United States,Spanish,Reggaeton,Solo,2013.0,125899.8,82189.3,43710.5,48887.0,77012.8,2026-07-01 00:00:00,2026.0
3,The Weeknd,Male,Canada,English,R&B,Solo,2009.0,95703.2,77189.4,18513.8,51020.0,44683.2,2026-07-01 00:00:00,2026.0
4,Justin Bieber,Male,Canada,English,Pop,Solo,2009.0,78104.4,48316.2,29788.2,29638.3,48466.1,2026-07-01 00:00:00,2026.0


In [ ]:
#PARA VER O DATASET INTEIRO:
#with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None):
#    display(raw_df)

## 2 · Duplicados

### 2.1 Procurar duplicados e removê-los
Verificar linhas inteiramente repetidas, `Artist ID` repetidos, e nomes de artista repetidos com
identificadores diferentes — são três situações distintas e nem todas são erros.


In [14]:
print("Linhas inteiramente duplicadas:", raw_df.duplicated().sum())
print("Artistas com ID duplicado:", raw_df.index.duplicated().sum())
print("Nomes de artistas repetidos:", raw_df['Artist Name'].duplicated().sum())

# Mostrar as linhas que estão duplicadas
duplicadas = raw_df[raw_df.duplicated(keep=False)]
display(duplicadas)

#Decisão: Remover cópias exatas e guardar num novo DataFrame de trabalho
df = raw_df.drop_duplicates().copy()
print(f"\nDimensões após remoção de duplicados exatos: {df.shape}")

Linhas inteiramente duplicadas: 10
Artistas com ID duplicado: 10
Nomes de artistas repetidos: 10


,Artist Name,Sex,Country of Origin,Primary Language,Primary Genre,Artist Type,Debut Year,Total Streams,Lead Streams,Feature Streams,Solo Streams,Collaborative Streams,Update Date,Year
Artist ID,,,,,,,,,,,,,,
45,Linkin Park,NaN,United States,English,Rock,Group,2000.0,33224.7,31426.6,1798.1,30151.2,3073.5,2026-07-01 00:00:00,2026.0
45,Linkin Park,NaN,United States,English,Rock,Group,2000.0,33224.7,31426.6,1798.1,30151.2,3073.5,2026-07-01 00:00:00,2026.0
140,Fleetwood Mac,NaN,United Kingdom,English,Rock,Group,1967.0,16577.5,16577.5,NaN,16567.4,10.1,2026-07-01 00:00:00,2026.0
140,Fleetwood Mac,NaN,United Kingdom,English,Rock,Group,1967.0,16577.5,16577.5,NaN,16567.4,10.1,2026-07-01 00:00:00,2026.0
218,John Mayer,Male,United States,English,Pop,Solo,1999.0,13208.3,12134.4,1076.8,11821.4,1386.9,2026-07-01 00:00:00,2026.0
218,John Mayer,Male,United States,English,Pop,Solo,1999.0,13208.3,12134.4,1076.8,11821.4,1386.9,2026-07-01 00:00:00,2026.0
221,System Of A Down,Male,United States,English,Metal,Group,1998.0,13009.4,13017.1,0.0,12963.5,45.9,2026-07-01 00:00:00,2026.0
221,System Of A Down,Male,United States,English,Metal,Group,1998.0,13009.4,13017.1,0.0,12963.5,45.9,2026-07-01 00:00:00,2026.0
263,Zedd,Male,Russia,English,EDM,Solo,2010.0,11481.4,11486.5,0.0,235.7,11245.7,2026-07-01 00:00:00,2026.0



Dimensões após remoção de duplicados exatos: (501, 14)


## 3 · Missing Values

### 3.1 Mapear os nulos
Quantos valores em falta tem cada coluna, em número absoluto e em percentagem.

### 3.2 Decidir coluna a coluna
A decisão depende da coluna e da proporção: apagar linhas, preencher, ou deixar em falta e assinalar.
Uma coluna com poucos nulos não se trata como uma com metade dos valores em falta.


In [19]:
#Valores em falta
nulos = pd.DataFrame({
    "Nulos": df.isnull().sum(),
    "Percentagem (%)": (df.isnull().mean() * 100).round(2)
})

print(nulos)

#Preencher nulos em 'Sex' para bandas/grupos onde não se aplica
df['Sex'] = df['Sex'].fillna('Group / N/A')

# Para colunas essenciais com poucos nulos (ex: Artist Name, Country), podemos inspecionar ou remover se irrecovery
print()
print("Linhas com nulos restantes:", df[df.isnull().any(axis=1)].shape[0])

                       Nulos  Percentagem (%)
Artist Name                1              0.2
Sex                        0              0.0
Country of Origin          1              0.2
Primary Language           1              0.2
Primary Genre              1              0.2
Artist Type                1              0.2
Debut Year                 1              0.2
Total Streams              6              1.2
Lead Streams               7              1.4
Feature Streams           10              2.0
Solo Streams               5              1.0
Collaborative Streams      2              0.4
Update Date                1              0.2
Year                       1              0.2

Linhas com nulos restantes: 26


## 4 · Outros problemas

### 4.1 Coerência das duas decomposições de streams
O guião garante que `Total Streams = Solo + Collaborative` e `Total Streams = Lead + Feature`.
Verificar as duas identidades e investigar as linhas em que não se confirmam.

### 4.2 Valores impossíveis
`Debut Year` fora de um intervalo razoável ou posterior a `Year`; streams negativos ou nulos;
`Update Date` com formatos inconsistentes.

### 4.3 Categorias escritas de várias maneiras
`Country of Origin`, `Primary Language`, `Primary Genre`, `Artist Type` e `Sex`: listar os valores
distintos de cada uma e procurar o mesmo valor escrito de formas diferentes (maiúsculas, acentos,
abreviaturas, espaços).


In [26]:
# 4.1 · Coerência das decomposições de streams
#Total = Solo + Collaborative
coerente_sc = np.isclose(df['Total Streams'], df['Solo Streams'] + df['Collaborative Streams'], equal_nan=True)
print(f"Inconsistências em Solo/Collab: {(~coerente_sc).sum()}")

#Total = Lead + Feature
coerente_lf = np.isclose(df['Total Streams'], df['Lead Streams'] + df['Feature Streams'], equal_nan=True)
print(f"Inconsistências em Lead/Feature: {(~coerente_lf).sum()}")

# 4.2 · Valores impossíveis (ex: Debut Year posterior ao ano de registo)
anos_invalidos = df[df['Debut Year'] > df['Year']]
print(f"Registos com ano de estreia posterior ao ano de referência: {len(anos_invalidos)}")

# 4.3 · Detetar inconsistências de escrita em colunas categóricas (str's)


Inconsistências em Solo/Collab: 10
Inconsistências em Lead/Feature: 415
Registos com ano de estreia posterior ao ano de referência: 0


## 5 · Conversões de tipo

Converter o que ficou com o tipo errado.


In [30]:
df['Debut Year'] = df['Debut Year'].astype('Int64')  # Converter anos para inteiro 
df['Year'] = df['Year'].astype('Int64')

df['Update Date'] = pd.to_datetime(df['Update Date']) # Converter Update Date para o tipo datetime

print("Tipos de dados atualizados:")
print(df.dtypes)

Tipos de dados atualizados:
Artist Name                         str
Sex                                 str
Country of Origin                   str
Primary Language                    str
Primary Genre                       str
Artist Type                         str
Debut Year                        Int64
Total Streams                   float64
Lead Streams                    float64
Feature Streams                 float64
Solo Streams                    float64
Collaborative Streams           float64
Update Date              datetime64[us]
Year                              Int64
dtype: object


## 6 · Perguntas sobre as datas de atualização

### 6.1 Definir data de referência
«*How many days ago*» depende de quando o notebook é executado. **Fixar uma data de referência,
escrevê-la explicitamente e justificar a escolha**, senão a resposta muda sozinha a cada execução.


### 6.2 (3a) Atualização mais recente e mais antiga
Há quantos dias ocorreram, e em que dia, mês e ano.

### 6.3 (3b) Registos desatualizados
Que artistas têm a última atualização **anterior ao fim de 2025** e, por isso, precisam de ver as
contagens atualizadas.


## 7 · Exportar e documentar

### 7.1 Gravar o dataset limpo
Exportar para `data/processed/artistas_limpo.csv`.
